# Farm Data Cleaning

## Required Input Files

The following files must be placed in the same directory as this notebook (`datascience/notebooks/`) before running. They are not committed to the repository due to privacy reasons and must be sourced from MS Teams.

| File | MS Teams Location |
|------|------------------|
| `farm_boundaries_sampled_20260320.gpkg` | MS Teams -> Planting Optimisation Tool -> Datasets -> `farm_boundaries_sampled_20260320.gpkg` |
| `DEM.tif` | MS Teams -> Planting Optimisation Tool -> Datasets -> GIS -> `DEM.tif` |
| `hotosm_tls_waterways_lines_gpkg.gpkg` | MS Teams -> Planting Optimisation Tool -> Datasets -> GIS -> Timor Leste Waterways -> `hotosm_tls_waterways_lines_gpkg.zip` (extract first) |

## Outputs

Running this notebook produces the following files (all gitignored, not committed):
- `farm_boundaries.csv` - raw export of the GeoPackage before cleaning
- `reports/farm_overlaps_report.csv` - farms with overlapping boundaries
- `reports/area_mismatch_report.csv` - farms where recorded area differs from geometry-derived area by >5%
- `farm_master.csv` - cleaned farm attributes
- `farm_boundaries_master.csv` - WKT geometries

`farm_master.csv` and `farm_boundaries_master.csv` have been committed to `backend/src/scripts/data/` and are used to populate the database via the standard workflow - you do not need to re-run this notebook unless the source dataset changes.

In [ ]:
# Standard library
import os

# Third-party libraries
import geopandas as gpd
import numpy as np
import rasterio
from rasterstats import zonal_stats
from shapely.geometry.base import BaseGeometry
from shapely.ops import transform
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# Build path to the GeoPackage
farm_file_path = "farm_boundaries_sampled_20260320.gpkg"

# Load the spatial farm boundary dataset
farm_boundaries = gpd.read_file(farm_file_path)

# Display column names
print(farm_boundaries.columns)

# Export the full dataset to CSV
csv_file_path = "farm_boundaries.csv"

farm_boundaries.to_csv(csv_file_path, index=False)

print(f"\nGeoPackage exported to CSV: {csv_file_path}")
# Check dataset structure, data types, and missing values
farm_boundaries.info()

#### Remove Privacy Information

In [2]:
# getrid of privacy info
cols_to_remove = ["name", "suku", "aldeia", "treeo2_id", "district", "subdistrict", "area"]
farm_cleaned = farm_boundaries.drop(columns=cols_to_remove, errors="ignore")

# Preview cleaned dataset
print("privacy info removed:")

# Check cleaned dataset info
farm_cleaned.info()

farm_cleaned.head()

privacy info removed:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3200 entries, 0 to 3199
Data columns (total 7 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   texture                           3153 non-null   object  
 1   ph                                3153 non-null   object  
 2   avg_annual_rainfall_2020-2024     3200 non-null   int32   
 3   avg_annual_temperature_2020-2024  3192 non-null   float64 
 4   elevation                         3200 non-null   int32   
 5   area_ha                           3200 non-null   float64 
 6   geometry                          3200 non-null   geometry
dtypes: float64(2), geometry(1), int32(2), object(2)
memory usage: 150.1+ KB


,texture,ph,avg_annual_rainfall_2020-2024,avg_annual_temperature_2020-2024,elevation,area_ha,geometry
0,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90..."
1,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90..."
2,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90..."
3,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90..."
4,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90..."


#### Rename Columns

In [3]:
farm_cleaned.insert(0, "farm_id", range(1, len(farm_cleaned) + 1))
farm_cleaned.rename(
    columns={
        "avg_annual_rainfall_2020-2024": "rainfall_mm",
        "avg_annual_temperature_2020-2024": "temperature_celsius",
        "elevation": "elevation_m",
        "texture": "soil_textures",
    },
    inplace=True,
)

# Check
print(farm_cleaned.columns)
print(farm_cleaned.head())

Index(['farm_id', 'soil_textures', 'ph', 'rainfall_mm', 'temperature_celsius',
       'elevation_m', 'area_ha', 'geometry'],
      dtype='object')
   farm_id soil_textures   ph  rainfall_mm  temperature_celsius  elevation_m  \
0        1          Clay  6.2         1959                 24.0          585   
1        2          Clay  6.2         1959                 24.0          481   
2        3    Sandy Loam  8.2         2021                 25.0          179   
3        4    Sandy Loam  5.9         2554                 24.0          260   
4        5          Clay  7.0         2021                 25.0          129   

   area_ha                                           geometry  
0     0.36  MULTIPOLYGON Z (((904783.597 9050919.352 0, 90...  
1     0.48  MULTIPOLYGON Z (((905235.474 9051056.651 0, 90...  
2     1.18  MULTIPOLYGON Z (((903414.205 9042747.721 0, 90...  
3     0.46  MULTIPOLYGON Z (((902007.836 9043097.668 0, 90...  
4     2.00  MULTIPOLYGON Z (((904016.568 9042595.439

#### Missing Values

In [4]:
farm_cleaned.isnull().sum()

farm_id                 0
soil_textures          47
ph                     47
rainfall_mm             0
temperature_celsius     8
elevation_m             0
area_ha                 0
geometry                0
dtype: int64

#### Data Validation after cleaning data

#### Check datatypes

In [5]:
print(farm_cleaned.dtypes)

farm_id                   int64
soil_textures            object
ph                       object
rainfall_mm               int32
temperature_celsius     float64
elevation_m               int32
area_ha                 float64
geometry               geometry
dtype: object


#### Valid Range Check

In [6]:
# Describe
print(farm_cleaned.describe())

           farm_id  rainfall_mm  temperature_celsius  elevation_m      area_ha
count  3200.000000  3200.000000          3192.000000  3200.000000  3200.000000
mean   1600.500000  1807.756562            24.194549   517.382187     1.413378
std     923.904757   347.589473             1.459383   254.967481     2.974257
min       1.000000   865.000000            19.000000     9.000000     0.000000
25%     800.750000  1529.000000            23.000000   353.000000     0.390000
50%    1600.500000  1734.000000            24.000000   489.500000     0.750000
75%    2400.250000  2034.000000            25.000000   711.000000     1.410000
max    3200.000000  2624.000000            28.000000  1313.000000    85.760000


### Spatial Consistency Check

#### Check missing geometries

In [6]:
missing_farm_boundaries = farm_boundaries.geometry.isnull().sum()
missing_farm_cleaned = farm_cleaned.geometry.isnull().sum()
print(missing_farm_boundaries)
print(missing_farm_cleaned)

0
0


#### Check for duplicate farm_id

In [ ]:
# Check duplicates in farm_cleaned (farm_boundaries gets farm_id assigned later in the area mismatch cell)
print(farm_cleaned["farm_id"].duplicated().sum())

#### Check Geometry validity

In [8]:
print(farm_boundaries.geometry.is_valid.value_counts())

True    3200
Name: count, dtype: int64


#### One-to-One 

In [ ]:
num_cleaned_records = len(farm_cleaned)
num_boundary_records = len(farm_boundaries)
one_to_one_issue = num_cleaned_records != num_boundary_records
print("One-to-one relationship :", one_to_one_issue, "\n")

#### Area Mismatch

In [10]:
# Reset index and assign farm_id
farm_boundaries = farm_boundaries.reset_index(drop=True)
farm_boundaries["farm_id"] = range(1, len(farm_boundaries) + 1)

# Ensure CRS in meters
farm_boundaries = farm_boundaries.to_crs(epsg=3857)

# Calculate geometry-derived area
farm_boundaries["calc_area_ha"] = farm_boundaries.geometry.area / 10000  # hectares

# Merge geometry-derived area into master
farm_cleaned = farm_cleaned.merge(farm_boundaries[["farm_id", "calc_area_ha"]], on="farm_id", how="left")

# Calculate absolute and percentage difference
farm_cleaned["abs_diff"] = (farm_cleaned["calc_area_ha"] - farm_cleaned["area_ha"]).abs()
farm_cleaned["pct_diff"] = (farm_cleaned["abs_diff"] / farm_cleaned["area_ha"]) * 100

# Report significant mismatches
area_mismatch = farm_cleaned[farm_cleaned["pct_diff"] > 5]
print(f"Number of farms with significant area mismatch (>5%): {len(area_mismatch)}")
print(area_mismatch[["farm_id", "area_ha", "calc_area_ha", "abs_diff", "pct_diff"]].head(10))

# Update area_ha where discrepancy >5%
farm_cleaned["area_ha"] = farm_cleaned.apply(lambda row: row["calc_area_ha"] if row["pct_diff"] > 5 else row["area_ha"], axis=1)

# Drop helper columns
farm_cleaned = farm_cleaned.drop(columns=["calc_area_ha", "abs_diff", "pct_diff"])

Number of farms with significant area mismatch (>5%): 75
     farm_id  area_ha  calc_area_ha  abs_diff   pct_diff
6          7     0.11      0.116325  0.006325   5.750152
216      217     0.24      0.252111  0.012111   5.046115
241      242     0.15      0.159124  0.009124   6.082805
244      245     0.01      0.012899  0.002899  28.993789
355      356     0.21      0.220670  0.010670   5.080890
516      517     0.27      0.283822  0.013822   5.119134
552      553     0.03      0.027198  0.002802   9.341491
560      561     0.14      0.148265  0.008265   5.903897
648      649     0.12      0.126593  0.006593   5.494446
657      658     0.08      0.085566  0.005566   6.958068


#### Overlap

In [11]:
# Spatial join to find intersecting geometries
joined = gpd.sjoin(farm_boundaries, farm_boundaries, how="inner", predicate="intersects")

# Remove self-matches
overlaps = joined[joined["farm_id_left"] != joined["farm_id_right"]]

# Keep only unique pairs
overlap_pairs = overlaps[["farm_id_left", "farm_id_right"]].copy()
overlap_pairs = overlap_pairs[overlap_pairs["farm_id_left"] < overlap_pairs["farm_id_right"]]

# Output
print("Number of overlapping boundaries:", len(overlaps))
print("Unique overlapping pairs:", len(overlap_pairs))
print(overlap_pairs.head())

Number of overlapping boundaries: 18
Unique overlapping pairs: 9
      farm_id_left  farm_id_right
2                3            756
450            451            452
1897          1898           2369
1899          1900           2580
1919          1920           2580


In [ ]:
report_folder = "reports"
os.makedirs(report_folder, exist_ok=True)

# Save overlaps report
overlap_pairs[["farm_id_left", "farm_id_right"]].to_csv(os.path.join(report_folder, "farm_overlaps_report.csv"), index=False)

# Save area mismatch report
area_mismatch[["farm_id", "area_ha", "calc_area_ha", "abs_diff", "pct_diff"]].to_csv(os.path.join(report_folder, "area_mismatch_report.csv"), index=False)

print("\nReports saved to:", report_folder)

#### Adding Lat, Lon and Spatial kNN Imputation for Numeric Fields

In [16]:
# Convert to GeoDataFrame if it's a plain DataFrame
if not isinstance(farm_cleaned, gpd.GeoDataFrame):
    farm_cleaned = gpd.GeoDataFrame(
        farm_cleaned,
        geometry="geometry",  # <-- replace with your geometry column name
        crs="EPSG:32751",  # <-- replace with your current CRS
    )

# Compute centroids in meters
centroids_proj = farm_cleaned.geometry.centroid

# If you want lon/lat, reproject back to EPSG:4326
farm_cleaned["longitude"] = centroids_proj.to_crs(epsg=4326).x
farm_cleaned["latitude"] = centroids_proj.to_crs(epsg=4326).y

# Show first 5 farms
farm_cleaned[["farm_id", "longitude", "latitude"]].head()
farm_cleaned.head()

,farm_id,soil_textures,ph,rainfall_mm,temperature_celsius,elevation_m,area_ha,geometry,longitude,latitude
0,1,Clay,6.2,1959,24.0,585,0.36,"MULTIPOLYGON Z (((904783.597 9050919.352 0, 90...",126.675704,-8.568798
1,2,Clay,6.2,1959,24.0,481,0.48,"MULTIPOLYGON Z (((905235.474 9051056.651 0, 90...",126.680149,-8.567578
2,3,Sandy Loam,8.2,2021,25.0,179,1.18,"MULTIPOLYGON Z (((903414.205 9042747.721 0, 90...",126.664651,-8.642259
3,4,Sandy Loam,5.9,2554,24.0,260,0.46,"MULTIPOLYGON Z (((902007.836 9043097.668 0, 90...",126.651458,-8.639690
4,5,Clay,7.0,2021,25.0,129,2.00,"MULTIPOLYGON Z (((904016.568 9042595.439 0, 90...",126.670084,-8.643001


In [17]:
# Spatial kNN Imputation for Numeric Fields

# Convert 'ph' column to float and round to 1 decimal
farm_cleaned["ph"] = farm_cleaned["ph"].astype(float).round(1)

coord_cols = ["latitude", "longitude"]

for col in ["ph", "temperature_celsius", "rainfall_mm", "elevation_m"]:
    train_df = farm_cleaned[farm_cleaned[col].notna()]
    test_df = farm_cleaned[farm_cleaned[col].isna()]

    if len(test_df) == 0:
        continue

    knn = KNeighborsRegressor(n_neighbors=5, weights="distance")
    knn.fit(train_df[coord_cols], train_df[col])

    farm_cleaned.loc[farm_cleaned[col].isna(), col] = knn.predict(test_df[coord_cols])

# Encode temporarily
farm_cleaned["soil_textures_encoded"] = farm_cleaned["soil_textures"].astype("category").cat.codes

mapping = dict(enumerate(farm_cleaned["soil_textures"].astype("category").cat.categories))

train_df = farm_cleaned[farm_cleaned["soil_textures"].notna()]
test_df = farm_cleaned[farm_cleaned["soil_textures"].isna()]

if len(test_df) > 0:
    knn_clf = KNeighborsClassifier(n_neighbors=5, weights="distance")
    knn_clf.fit(train_df[coord_cols], train_df["soil_textures_encoded"])

    preds = knn_clf.predict(test_df[coord_cols])

    farm_cleaned.loc[farm_cleaned["soil_textures"].isna(), "soil_textures"] = [mapping[p] for p in preds]

farm_cleaned.drop(columns=["soil_textures_encoded"], inplace=True)

# Check missing values again
print(farm_cleaned.isnull().sum())
print(farm_cleaned.columns)

farm_id                0
soil_textures          0
ph                     0
rainfall_mm            0
temperature_celsius    0
elevation_m            0
area_ha                0
geometry               0
longitude              0
latitude               0
dtype: int64
Index(['farm_id', 'soil_textures', 'ph', 'rainfall_mm', 'temperature_celsius',
       'elevation_m', 'area_ha', 'geometry', 'longitude', 'latitude'],
      dtype='object')


#### Spatial kNN Imputation for Soil Texture

In [18]:
# Standardize texture labels
farm_cleaned["soil_textures"] = farm_cleaned["soil_textures"].str.lower().str.strip()
farm_cleaned["soil_textures"] = farm_cleaned["soil_textures"].replace({"silt loam": "silty loam", "silt clay": "silty clay"})

# Project schema mapping
soil_textures_map = {
    "sand": 1,
    "loamy sand": 2,
    "sandy loam": 3,
    "loam": 4,
    "silty loam": 5,
    "silt": 6,
    "sandy clay loam": 7,
    "clay loam": 8,
    "silty clay loam": 9,
    "sandy clay": 10,
    "silty clay": 11,
    "clay": 12,
}
valid_textures = list(soil_textures_map.keys())

# Set invalid textures to NaN
farm_cleaned.loc[~farm_cleaned["soil_textures"].isin(valid_textures), "soil_textures"] = np.nan

# Spatial kNN imputation for texture
coord_cols = ["latitude", "longitude"]
train_df = farm_cleaned[farm_cleaned["soil_textures"].notna()].copy()
test_df = farm_cleaned[farm_cleaned["soil_textures"].isna()].copy()

if len(test_df) > 0:
    # Encode textures as integers
    train_df["soil_textures_encoded"] = train_df["soil_textures"].astype("category").cat.codes
    mapping = dict(enumerate(train_df["soil_textures"].astype("category").cat.categories))

    knn_clf = KNeighborsClassifier(n_neighbors=5, weights="distance")
    knn_clf.fit(train_df[coord_cols], train_df["soil_textures_encoded"])

    preds = knn_clf.predict(test_df[coord_cols])
    farm_cleaned.loc[farm_cleaned["soil_textures"].isna(), "soil_textures"] = [mapping[p] for p in preds]

# Map to IDs
farm_cleaned["soil_texture_id"] = farm_cleaned["soil_textures"].map(soil_textures_map)

# Check results
print(farm_cleaned[["soil_textures", "soil_texture_id"]].drop_duplicates())
print("Missing soil_texture_id values:", farm_cleaned["soil_texture_id"].isnull().sum())

     soil_textures  soil_texture_id
0             clay               12
2       sandy loam                3
11       clay loam                8
25            loam                4
530     silty loam                5
561     sandy clay               10
2351    silty clay               11
Missing soil_texture_id values: 0


### Core Engineered features

#### Compute Coastal Environment Flag

In [ ]:
farm_cleaned["coastal"] = (farm_cleaned["elevation_m"] < 100) & (farm_cleaned["rainfall_mm"].between(500, 3000))
farm_cleaned.head()

#### Finding Slope

In [ ]:
print("Farms CRS:", farm_cleaned.crs)

dem_path = "DEM.tif"

with rasterio.open(dem_path) as dem:
    print("DEM CRS:", dem.crs)

    dem_data = dem.read(1)
    dem_transform = dem.transform
    dem_nodata = dem.nodata

# Reproject farms to EPSG:3857 (to match DEM)
farm_cleaned = farm_cleaned.to_crs(epsg=3857)

print("Reprojected farms CRS:", farm_cleaned.crs)

# Convert DEM to float & handle nodata
dem_float = dem_data.astype(float)
dem_float[dem_float == dem_nodata] = np.nan

# Resolution
dem_res_x = dem_transform.a
dem_res_y = abs(dem_transform.e)

# Gradient
dx, dy = np.gradient(dem_float, dem_res_x, dem_res_y)

# Slope in degrees
slope_degrees = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))

# Zonal statistics
slope_stats = zonal_stats(farm_cleaned.geometry, slope_degrees, affine=dem_transform, stats=["mean"], nodata=np.nan, all_touched=True)

farm_cleaned["slope"] = [s["mean"] for s in slope_stats]
print(farm_cleaned[["farm_id", "slope"]].head(10))
farm_cleaned.head()

#### Riparian

In [ ]:
# Load waterways
waterways_path = "hotosm_tls_waterways_lines_gpkg.gpkg"
waterways = gpd.read_file(waterways_path)
waterways = waterways.to_crs(farm_cleaned.crs)

# Create 15m buffer
waterways_buffer = waterways.buffer(15)
waterways_buffer_gdf = gpd.GeoDataFrame(geometry=waterways_buffer)

# Spatial join: which farms intersect any buffer
joined = gpd.sjoin(farm_cleaned, waterways_buffer_gdf, how="left", predicate="intersects")

# Initialize column as False
farm_cleaned["riparian"] = False

# Mark True only where intersection exists
intersecting_farms = joined[joined["index_right"].notna()].index.unique()
farm_cleaned.loc[intersecting_farms, "riparian"] = True

print(farm_cleaned["riparian"].value_counts())

# Check
print(farm_cleaned[["farm_id", "riparian"]].head(10))

#### convert geometry to 2D

In [29]:
print("CRS before 2D conversion:", farm_cleaned.crs)


def force_2d_safe(geom):
    # If geom is None or empty, return None
    if geom is None or geom.is_empty:
        return None
    # Only apply transform if geom is a shapely geometry
    if isinstance(geom, BaseGeometry):
        return transform(lambda x, y, z=None: (x, y), geom)
    else:
        return None  # fallback for invalid entries


# Apply safe 2D conversion
farm_cleaned["geometry"] = farm_cleaned["geometry"].apply(force_2d_safe)

print("CRS after 2D conversion:", farm_cleaned.crs)
farm_cleaned = farm_cleaned.to_crs(epsg=4326)
print(farm_cleaned.crs)

CRS before 2D conversion: EPSG:3857
CRS after 2D conversion: EPSG:3857
EPSG:4326


#### Range Checks

In [30]:
# Define correct ranges according to the data dictionary
valid_ranges = {"ph": (4.0, 8.5), "temperature_celsius": (15, 30), "rainfall_mm": (500, 3000), "elevation_m": (0, 2963), "area_ha": (0, 100)}

# Perform range check
for col, (min_val, max_val) in valid_ranges.items():
    if col in farm_cleaned.columns:
        outliers = farm_cleaned[(farm_cleaned[col] < min_val) | (farm_cleaned[col] > max_val)]
        print(f"\n--- {col} ---")
        print(f"Out-of-range count: {len(outliers)}")
        print(outliers[[col]].head())


--- ph ---
Out-of-range count: 0
Empty DataFrame
Columns: [ph]
Index: []

--- temperature_celsius ---
Out-of-range count: 0
Empty DataFrame
Columns: [temperature_celsius]
Index: []

--- rainfall_mm ---
Out-of-range count: 0
Empty DataFrame
Columns: [rainfall_mm]
Index: []

--- elevation_m ---
Out-of-range count: 0
Empty DataFrame
Columns: [elevation_m]
Index: []

--- area_ha ---
Out-of-range count: 0
Empty DataFrame
Columns: [area_ha]
Index: []


#### Placeholders

In [31]:
placeholder_fields = ["nitrogen_fixing", "shade_tolerant", "bank_stabilising"]

for field in placeholder_fields:
    farm_cleaned[field] = False

#### Export to CSV

In [ ]:
# Enforce types and precision to match backend schema
farm_export = farm_cleaned.copy()
farm_export["rainfall_mm"] = farm_export["rainfall_mm"].astype(int)
farm_export["temperature_celsius"] = farm_export["temperature_celsius"].astype(int)
farm_export["elevation_m"] = farm_export["elevation_m"].astype(int)
farm_export["area_ha"] = farm_export["area_ha"].round(3)
farm_export["latitude"] = farm_export["latitude"].round(5)
farm_export["longitude"] = farm_export["longitude"].round(5)
farm_export["slope"] = farm_export["slope"].round(2)

# Rename farm_id to external_id and drop the raw soil texture string
farm_export = farm_export.rename(columns={"farm_id": "external_id"})
farm_export = farm_export.drop(columns=["soil_textures", "geometry"], errors="ignore")

# Export farms CSV
farm_export.to_csv("farm_master.csv", index=False)
print(f"Farms exported: {len(farm_export)} rows")
print(farm_export.columns.tolist())

# Export boundaries CSV (external_id + WKT geometry for import_boundaries.py)
boundaries_export = farm_cleaned[["farm_id", "geometry"]].copy()
boundaries_export = boundaries_export.rename(columns={"farm_id": "external_id"})
boundaries_export["boundary"] = boundaries_export["geometry"].apply(lambda g: g.wkt)
boundaries_export = boundaries_export.drop(columns=["geometry"])
boundaries_export.to_csv("farm_boundaries_master.csv", index=False)
print(f"Boundaries exported: {len(boundaries_export)} rows")

In [33]:
print(farm_cleaned.head())

   farm_id soil_textures   ph  rainfall_mm  temperature_celsius  elevation_m  \
0        1          clay  6.2         1959                 24.0          585   
1        2          clay  6.2         1959                 24.0          481   
2        3    sandy loam  8.2         2021                 25.0          179   
3        4    sandy loam  5.9         2554                 24.0          260   
4        5          clay  7.0         2021                 25.0          129   

   area_ha                                           geometry   longitude  \
0     0.36  MULTIPOLYGON (((126.67605 -8.56847, 126.67613 ...  126.675704   
1     0.48  MULTIPOLYGON (((126.68013 -8.56719, 126.68065 ...  126.680149   
2     1.18  MULTIPOLYGON (((126.66434 -8.64235, 126.66422 ...  126.664651   
3     0.46  MULTIPOLYGON (((126.65155 -8.63931, 126.65155 ...  126.651458   
4     2.00  MULTIPOLYGON (((126.66981 -8.64367, 126.66968 ...  126.670084   

   latitude  soil_texture_id  coastal      slope  ripari